# 03 — Investigasi `sks_sem2` Dominance

**Masalah:** Di stratified baseline, `sks_sem2` punya feature importance **0.621** — mendominasi semua fitur lain. Root split ada di `sks_sem2 <= 18.50`.

**Pertanyaan:** Apakah ini genuine signal atau artifact dari bug TSKS (SKS kumulatif vs per-semester)?

**Rencana:**
1. Distribusi `sks_sem2` per target class — genuine separation?
2. Deteksi nilai abnormal — range, outlier, per angkatan
3. Tracing TSKS bug — bandingkan raw vs clean, cek korelasi antar semester
4. Decision rule analysis — distribusi di sekitar root split threshold
5. Kesimpulan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load both datasets
df_clean = pd.read_csv('../3-data-preparation/dataset_clean.csv')
df_raw   = pd.read_csv('../3-data-preparation/dataset.csv')

print("Dataset loaded.")
print(f"  Clean: {df_clean.shape}")
print(f"  Raw:   {df_raw.shape}")

## Step 1: Distribusi `sks_sem2` per Target Class

In [ ]:
print("=== sks_sem2 descriptive stats per target ===\n")
print(df_clean.groupby('target')['sks_sem2'].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Boxplot
sns.boxplot(x='target', y='sks_sem2', data=df_clean, ax=axes[0])
axes[0].set_title('Boxplot: sks_sem2 by Target')
axes[0].set_xticklabels(['Tidak Tepat', 'Tepat Waktu'])

# Histogram
for target_val, label, color in [(0, 'Tidak Tepat', 'orange'), (1, 'Tepat Waktu', 'steelblue')]:
    subset = df_clean[df_clean['target'] == target_val]
    axes[1].hist(subset['sks_sem2'], bins=20, alpha=0.6, label=label, color=color, density=True)
axes[1].set_title('Histogram: sks_sem2 Density')
axes[1].set_xlabel('sks_sem2')
axes[1].legend()

# Violin
sns.violinplot(x='target', y='sks_sem2', data=df_clean, ax=axes[2])
axes[2].set_title('Violin: sks_sem2 by Target')
axes[2].set_xticklabels(['Tidak Tepat', 'Tepat Waktu'])

plt.tight_layout()
plt.show()

# Target rate per sks_sem2 quartile
print("\n=== Target rate per sks_sem2 quartile ===")
df_clean['sks_quartile'] = pd.qcut(df_clean['sks_sem2'], q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
print(df_clean.groupby('sks_quartile', observed=False)['target'].agg(['count', 'mean']).round(3))

## Step 2: Deteksi Outlier & Nilai Abnormal

In [ ]:
# Value counts — head and tail
vc = df_clean['sks_sem2'].value_counts().sort_index()
print("=== 10 smallest values ===")
print(vc.head(10))
print(f"\n=== 10 largest values ===")
print(vc.tail(10))

# Per-angkatan stats
print("\n=== sks_sem2 per angkatan ===\n")
angk_stats = df_clean.groupby('angkatan')['sks_sem2'].describe().round(2)
angk_stats['neg_rate'] = df_clean.groupby('angkatan')['target'].apply(lambda x: (x==0).mean()).round(3)
print(angk_stats)

# Check: any value > 24 (impossible for 1 normal semester)?
abnormal = df_clean[df_clean['sks_sem2'] > 24]
print(f"\n=== sks_sem2 > 24 (abnormal): {len(abnormal)} rows ===")
if len(abnormal) > 0:
    print(abnormal[['angkatan', 'sks_sem2', 'sks_sem1', 'sks_sem3', 'target']])

# Check: any value < 2?
low = df_clean[df_clean['sks_sem2'] < 2]
print(f"\n=== sks_sem2 < 2 (suspicious): {len(low)} rows ===")
if len(low) > 0:
    print(low[['angkatan', 'sks_sem2', 'sks_sem1', 'sks_sem3', 'target']].head(10))

## Step 3: Tracing TSKS Bug — Raw vs Clean

In [ ]:
# Compare raw vs clean sks_sem2
# In raw, check if sks_sem2 column exists
if 'sks_sem2' in df_raw.columns:
    comparison = pd.DataFrame({
        'angkatan': df_raw['angkatan'],
        'target': df_raw['target'],
        'raw_sks_sem2': df_raw['sks_sem2'],
        'clean_sks_sem2': df_clean['sks_sem2']
    })
    
    # How many were imputed? (raw had NULL or 0 → clean has imputed value)
    comparison['was_imputed'] = (comparison['raw_sks_sem2'].isna()) | (comparison['raw_sks_sem2'] == 0)
    
    n_imputed = comparison['was_imputed'].sum()
    n_total = len(comparison)
    print(f"Imputed sks_sem2: {n_imputed} / {n_total} ({n_imputed/n_total*100:.1f}%)")
    
    if n_imputed > 0:
        print(f"\nImputed rows — target distribution:")
        print(comparison[comparison['was_imputed']]['target'].value_counts())
        print(f"\nImputed rows — angkatan distribution:")
        print(comparison[comparison['was_imputed']]['angkatan'].value_counts().sort_index())
        
    # Before/after stats per target
    print(f"\n=== sks_sem2: Raw vs Clean per target ===")
    for t in [0, 1]:
        raw_sub = comparison[comparison['target'] == t]['raw_sks_sem2'].dropna()
        clean_sub = comparison[comparison['target'] == t]['clean_sks_sem2']
        print(f"\nTarget={t} ({'Tidak Tepat' if t==0 else 'Tepat Waktu'}):")
        print(f"  Raw   : mean={raw_sub.mean():.1f}, median={raw_sub.median():.1f}, "
              f"non-null={len(raw_sub)}")
        print(f"  Clean : mean={clean_sub.mean():.1f}, median={clean_sub.median():.1f}")
else:
    print("sks_sem2 not in raw dataset columns")
    print(f"Raw columns: {list(df_raw.columns)}")

# Check inter-semester SKS correlation
print(f"\n=== SKS inter-semester correlation ===")
sks_corr = df_clean[['sks_sem1', 'sks_sem2', 'sks_sem3']].corr().round(3)
print(sks_corr)

# Check: cumulative pattern? (sks_sem2 always >= sks_sem1 would be suspicious)
cumulative = (df_clean['sks_sem2'] >= df_clean['sks_sem1']).sum()
print(f"\nsks_sem2 >= sks_sem1: {cumulative} / {len(df_clean)} ({cumulative/len(df_clean)*100:.1f}%)")
print("(Jika >80%: mungkin masih ada residual cumulative pattern)")

## Step 4: Decision Rule Analysis — Root Split

In [ ]:
# Analyze the root split threshold: sks_sem2 <= 18.50
threshold = 18.50

left = df_clean[df_clean['sks_sem2'] <= threshold]
right = df_clean[df_clean['sks_sem2'] > threshold]

print(f"Root split: sks_sem2 <= {threshold}")
print(f"{'='*55}")
print(f"Left  (<= {threshold}): {len(left)} rows ({len(left)/len(df_clean)*100:.1f}%)")
print(f"  Target: {dict(left['target'].value_counts())}")
print(f"  Neg rate: {left['target'].mean():.3f}")  
print(f"  sks_sem2 range: [{left['sks_sem2'].min()}, {left['sks_sem2'].max()}]")
print()
print(f"Right (> {threshold}): {len(right)} rows ({len(right)/len(df_clean)*100:.1f}%)")
print(f"  Target: {dict(right['target'].value_counts())}")
print(f"  Neg rate: {right['target'].mean():.3f}")
print(f"  sks_sem2 range: [{right['sks_sem2'].min()}, {right['sks_sem2'].max()}]")

# Target rate per angkatan in right branch
print(f"\nRight branch (> {threshold}) breakdown per angkatan:")
for ang in sorted(right['angkatan'].unique()):
    sub = right[right['angkatan'] == ang]
    print(f"  {ang}: {len(sub)} rows, target mean={sub['target'].mean():.3f}")

# Where do the "Tidak Tepat" students in the left branch go?
print(f"\nLeft branch (<= {threshold}) — 'Tidak Tepat' students:")
left_neg = left[left['target'] == 0]
print(f"  Count: {len(left_neg)}")
if len(left_neg) > 0:
    print(f"  ips_sem1 stats: mean={left_neg['ips_sem1'].mean():.2f}, "
          f"median={left_neg['ips_sem1'].median():.2f}")
    # Next split: ips_sem1 <= 2.70
    below = left_neg[left_neg['ips_sem1'] <= 2.70]
    above = left_neg[left_neg['ips_sem1'] > 2.70]
    print(f"  ips_sem1 <= 2.70: {len(below)} (predicted TIDAK TEPAT by next split)")
    print(f"  ips_sem1 > 2.70:  {len(above)} (continue to deeper branches)")

## Step 5: Kesimpulan

In [ ]:
print("=== FINAL ASSESSMENT: sks_sem2 ===\n")

# Gather evidence
cleaned_mean_diff = df_clean.groupby('target')['sks_sem2'].mean().diff().iloc[-1]
sks_corr_with_target = df_clean['sks_sem2'].corr(df_clean['target'])
raw_vs_clean_check = 'sks_sem2' in df_raw.columns

print(f"1. Mean difference (Tepat - Tidak): {cleaned_mean_diff:.2f} SKS")
print(f"   {'✓ Separation jelas' if abs(cleaned_mean_diff) > 5 else '? Separation marginal' if abs(cleaned_mean_diff) > 2 else '✗ No separation'}")
print()
print(f"2. Pearson r with target: {sks_corr_with_target:.4f}")
print(f"   {'✓ Moderate-to-strong' if abs(sks_corr_with_target) > 0.3 else '? Weak-to-moderate' if abs(sks_corr_with_target) > 0.1 else '✗ Near zero'}")
print()
print(f"3. Raw vs Clean comparison: {'Available' if raw_vs_clean_check else 'Raw data not loaded'}")

# Summary judgment
print(f"\n{'='*55}")
print("JUDGMENT:")
print(f"{'='*55}")

# Check if there are abnormal values
abnormal_count = len(df_clean[df_clean['sks_sem2'] > 24])
if abnormal_count > 0:
    print(f"✗ ARTIFACT: {abnormal_count} rows with sks_sem2 > 24")
elif abs(cleaned_mean_diff) > 3 and abs(sks_corr_with_target) > 0.2:
    print("✓ GENUINE SIGNAL: Clear separation between classes")
    print("  sks_sem2 is a legitimate predictor — likely reflects course load reduction by struggling students")
else:
    print("? MIXED: Some separation but needs more context")
    print("  Check the visualizations above for detailed patterns")